# Stage 3 · RAG over the AI Media Dataset — project skeleton

*HSLU Computational Language Technologies · capstone project · built on `anatoolbox`*

This notebook is the skeleton of your Stage 3 project. It runs from top to bottom as it is, and
at every step it gives you a **baseline** and a way to **measure** it. Going beyond the baselines
is your job.

It follows the structure of Chapter 7 of *The Art of AI Product Development* (recommended
reading for Stage 3):

| Part | Chapter | What you do |
|---|---|---|
| **A · Semantic search** | 7.2 | build the search index · evaluate retrieval · optimize retrieval |
| **B · End-to-end RAG** | 7.3 | set up answer generation · evaluate answers · optimize the system, including your Stage 1 knowledge graph |
| **C · Results** | — | compare every configuration · one limitation → one enhancement |

### How to read the cells

| Marker | Meaning |
|---|---|
| 🟦 **Baseline** | A standard method, ready to run. The reference point for everything else. |
| 📏 **Measure** | Evaluation of whatever ran just before. Every result lands in a comparison table. |
| 🟩 **Worked example** | One optimization implemented completely, to show how it is done. |
| ✏️ **Your turn** | A template that runs unchanged (it reproduces the baseline) until you change it. |

Run the whole notebook once before changing anything: that gives you the baseline numbers
your changes will be compared with.

### Team and disclosure

*Required for the submission — fill in before handing in.*

| Team member | Contribution |
|---|---|
| … | … |

**AI coding tools used:** *name each tool, and say how you used it.*

## 0 · Setup

### How this notebook uses `anatoolbox`

Every step calls a **tool**: `chunk_by_size`, `retrieve_passages`, `synthesize_answer`, …
Each tool is a baseline. Every tool result carries `provenance` (which tool, with which
settings, derived from which earlier results), so each number in the final tables traces
back to the configuration that produced it.

To try your own method, there are three routes, from least to most code:

1. **Plug in a function** — e.g. your Stage 2 embedding model: `configure_embedder(my_model.encode)`.
2. **Subclass a tool and override one hook** — the step you want to change. Everything else
   (inputs, chaining, provenance) is inherited:

   ```python
   class ChunkBySentence(ChunkBySizeTool):
       tool_name = "chunk_by_sentence"          # a new name, same prefix
       def split(self, text, settings):         # the one step you change
           ...
   ```
3. **Write a new tool** when no shipped tool does the kind of work you need.

`docs/extending.md` in the anatoolbox repository lists every hook. `show_hooks(...)` below
prints them for a tool.

In [ ]:
# In Colab, install the toolbox first, the way your course instructions describe, e.g.:
# %pip install -q "anatoolbox[embeddings]" pandas

import importlib.util
import inspect
import json
import os
import platform
import time
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

import anatoolbox
from anatoolbox import ToolContext, resolve_tools

pd.set_option("display.max_colwidth", 120)
anatoolbox.register_reference_tools()
ctx = ToolContext()  # a notebook needs no memory layer: results are passed from tool to tool

(ingest, ingest_graph, chunk, rewrite, retrieve, rerank, synthesize,
 draft_questions, retrieval_metrics, judge) = resolve_tools([
    "ingest_corpus", "ingest_knowledge_graph", "chunk_by_size", "rewrite_query_for_retrieval",
    "retrieve_passages", "rerank_passages", "synthesize_answer",
    "extract_test_questions", "calculate_retrieval_metrics", "score_rag_answer",
])


def show_hooks(tool_class):
    # The methods a subclass of `tool_class` can override, with the first line of their docs.
    own = [name for name, member in vars(tool_class).items()
           if inspect.isfunction(member) and not name.startswith("_") and name not in ("run", "render")]
    for name in own:
        doc = (inspect.getdoc(getattr(tool_class, name)) or "").splitlines()
        print(f"  {name}{inspect.signature(getattr(tool_class, name))}\n      {doc[0] if doc else ''}")


print("python     :", platform.python_version())
print("anatoolbox :", anatoolbox.__version__)
print("dense retrieval and reranking:", "available" if importlib.util.find_spec("sentence_transformers") else "install sentence-transformers")

### Project settings

Everything you are likely to change between runs is in this cell. The defaults are sized for
a full run on Colab. For a quick check, lower `max_articles`, `qa_passages` and `rag_eval_questions`.

In [ ]:
CONFIG = {
    "track": "Agentic Web",          # "Hardware & Infrastructure" | "Foundation Models" | "Agentic Web"
    "max_articles": None,            # None = every article of the track; a number = an even sample over time
    "qa_passages": 120,              # passages to draft Q&A pairs from (the brief asks for ~100–150)
    "questions_per_passage": 2,      # 1–3, for ~200–300 Q&A pairs
    "rag_eval_questions": 50,        # questions per RAG configuration in part B (each costs two LLM calls)
    # Language models: any OpenAI-compatible endpoint. The brief requires a DIFFERENT model to
    # generate your Q&A pairs (and, here, to judge answers) than the one inside your RAG system.
    "base_url": None,                # None = OpenAI; e.g. "https://openrouter.ai/api/v1" or "http://localhost:11434/v1"
    "api_key": os.environ.get("OPENAI_API_KEY"),
    "rag_model": None,               # ← e.g. "gpt-4o-mini" — writes rewrites and answers
    "evaluation_model": None,        # ← a different, preferably stronger model — drafts Q&A pairs, judges answers
}
# Quick test runs can override settings without editing this cell:
CONFIG.update(json.loads(os.environ.get("COURSE_NOTEBOOK_OVERRIDES", "{}")))

DATA_DIR = Path(os.environ.get("AI_MEDIA_DATA_DIR", "data"))
RESULTS_DIR = DATA_DIR / "results" / CONFIG["track"].lower().replace(" & ", "_").replace(" ", "_")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(json.dumps({k: v for k, v in CONFIG.items() if k != "api_key"}, indent=2))

In [ ]:
from anatoolbox.llm_client import call_llm_text, configure_llm, model_for

configure_llm(
    base_url=CONFIG["base_url"],
    api_key=CONFIG["api_key"],
    models={role: name for role, name in {"default": CONFIG["rag_model"], "evaluation": CONFIG["evaluation_model"]}.items() if name},
)


def check_model(role):
    try:
        started = time.perf_counter()
        call_llm_text("Reply with one word.", "Say: ready", model=model_for(role), max_tokens=5)
        return True, f"{model_for(role)} — responding ({time.perf_counter() - started:.1f} s)"
    except Exception as exc:  # no model configured, unreachable endpoint, bad key, ...
        return False, f"not available — {type(exc).__name__}: {str(exc)[:150]}"


LLM_READY, rag_status = check_model("default")
EVAL_READY, eval_status = check_model("evaluation")
print("RAG model        :", rag_status)
print("evaluation model :", eval_status)
if LLM_READY and EVAL_READY and model_for("default") == model_for("evaluation"):
    print("\n⚠ The evaluation model is the RAG model. Fine for a test run — not for your submission.")
if not LLM_READY:
    print("\nSet rag_model and evaluation_model in CONFIG. Until then, the cells that need a model are skipped.")

## 1 · Data: your track, and your Stage 1 knowledge graph

The AI Media Dataset downloads from Kaggle's public API on the first run (~58 MB, no account
needed) and is cached in `data/`. Stage 3 works on the articles of **your track**. The keyword
patterns below are a rough stand-in: replace them with the track definition you settled on in
Stage 1.

In [ ]:
import io
import urllib.request
import zipfile

KAGGLE_URL = "https://www.kaggle.com/api/v1/datasets/download/jannalipenkova/ai-media-dataset"
TRACKS = {  # ← replace with your Stage 1 track definition
    "Hardware & Infrastructure": r"\bgpus?\b|accelerator|nvidia|data cent(?:er|re)|semiconductor|\btsmc\b|\bchips?\b",
    "Foundation Models": r"foundation model|large language model|\bllms?\b|gpt-?\d|\bllama\b|gemini|claude|mistral|\bqwen\b",
    "Agentic Web": r"ai agents?|agentic|\bmcp\b|model context protocol|agent2agent|\ba2a\b|browser agent|computer use",
}


def find_or_download_csv():
    explicit = os.environ.get("AI_MEDIA_CSV")
    if explicit and Path(explicit).exists():
        return Path(explicit)
    cached = sorted(DATA_DIR.glob("ai_media_dataset_*.csv"))
    if cached:
        return cached[-1]
    print("Downloading the AI Media Dataset from Kaggle …")
    with urllib.request.urlopen(KAGGLE_URL, timeout=180) as response:
        payload = response.read()
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        name = next(n for n in archive.namelist() if n.endswith(".csv"))
        archive.extract(name, DATA_DIR)
    return DATA_DIR / name


df = pd.read_csv(find_or_download_csv())
haystack = (df["title"] + " " + df["content"] + " " + df["tags"]).str.lower()
track_df = df[haystack.str.contains(TRACKS[CONFIG["track"]], regex=True)].sort_values("date")
if CONFIG["max_articles"] and len(track_df) > CONFIG["max_articles"]:
    step = len(track_df) / CONFIG["max_articles"]
    track_df = track_df.iloc[[int(i * step) for i in range(CONFIG["max_articles"])]]
track_csv = RESULTS_DIR / "track_articles.csv"
track_df.to_csv(track_csv, index=False)

articles = ingest.run({"path": str(track_csv), "name": "track_articles", "text_field": "content", "id_field": "Unnamed: 0"}, context=ctx)
print(f"{CONFIG['track']}: {articles['records']:,} articles, {track_df['date'].min()} → {track_df['date'].max()}")

### Your Stage 1 knowledge graph

Stage 1 ends with a knowledge graph and its schema, stored for reuse. Load them here.
`ingest_knowledge_graph` reads an **edge table** (CSV, TSV, JSON, JSONL) with one row per fact —
`subject`, `relation`, `object`, and optionally `subject_type`, `object_type`, `source_ids`
(the article ids a fact comes from) and `date` — or **NetworkX node-link JSON**
(`networkx.node_link_data(G)`). Columns with other names can be mapped with
`subject_field=`, `relation_field=`, `object_field=`.

Keep `source_ids`: they make graph evidence traceable to articles, which the brief requires.

**Until your file is in place, a placeholder graph is used:** co-mentions of a handful of
hand-picked names. It exists only so the notebook runs end to end — it is *not* a Stage 1 graph,
and results with it say nothing about graph-enhanced RAG.

In [ ]:
from anatoolbox.graph import KnowledgeGraph, get_graph, register_graph

GRAPH_FILE = DATA_DIR / "stage1_graph_edges.csv"   # ← your Stage 1 export
SCHEMA_FILE = DATA_DIR / "stage1_schema.md"         # ← your schema / data dictionary

if GRAPH_FILE.exists():
    loaded = ingest_graph.run({"path": str(GRAPH_FILE), **({"schema_path": str(SCHEMA_FILE)} if SCHEMA_FILE.exists() else {})}, context=ctx)
    GRAPH = get_graph(loaded["graph"])
    GRAPH_IS_PLACEHOLDER = False
else:
    import itertools
    import re

    PLACEHOLDER_NAMES = {
        "Agentic Web": ["OpenAI", "Anthropic", "Google", "Microsoft", "Amazon", "Salesforce", "Perplexity", "MCP", "A2A", "Operator", "Claude", "Gemini", "Copilot", "Agentforce"],
        "Hardware & Infrastructure": ["Nvidia", "AMD", "Intel", "TSMC", "Broadcom", "Google", "Microsoft", "Amazon", "Meta", "Blackwell", "H100", "H20", "CoreWeave", "OpenAI"],
        "Foundation Models": ["OpenAI", "Anthropic", "Google", "Meta", "Mistral", "DeepSeek", "Alibaba", "GPT-4o", "GPT-5", "Claude", "Gemini", "Llama", "Qwen", "Grok"],
    }[CONFIG["track"]]
    pairs = {}
    for row in track_df.itertuples():
        text = f"{row.title} {row.content}"
        named = sorted({n for n in PLACEHOLDER_NAMES if re.search(rf"(?<!\w){re.escape(n)}(?!\w)", text)})
        for a, b in itertools.combinations(named, 2):
            pairs.setdefault((a, b), []).append(str(getattr(row, "_1")))
    GRAPH = register_graph(KnowledgeGraph.from_records(
        [{"subject": a, "relation": "co_mentioned_with", "object": b, "weight": len(ids), "source_ids": ids[:20]}
         for (a, b), ids in pairs.items() if len(ids) >= 3],
        name="placeholder_graph",
    ))
    GRAPH_IS_PLACEHOLDER = True
    print(f"⚠ No Stage 1 graph at {GRAPH_FILE.name} in the data folder — using the placeholder graph.\n")

print(json.dumps(GRAPH.describe(), indent=2)[:900])

---
# Part A · Semantic search (Chapter 7.2)

Retrieval decides what your answers can be based on: a passage that is not retrieved cannot be
cited. Part A builds the search index, evaluates it, and optimizes it — in that order, so every
optimization is measured against the baseline.

## A1 · Build the embedding database (7.2.2)

### 🟦 Baseline: fixed-size chunks

Embedding models read a few hundred tokens; articles are far longer. `chunk_by_size` cuts every
article into windows of `size` words that overlap by `overlap` words — ignoring sentences,
paragraphs and meaning. That is what makes it a baseline.

In [ ]:
from anatoolbox.corpus import get_corpus

chunks = chunk.run({"input": articles, "size": 200, "overlap": 40}, context=ctx)
print(f"{chunks['chunks']:,} chunks from {chunks['articles']:,} articles · words per chunk: {chunks['tokens']}")
display(pd.DataFrame(get_corpus(chunks["corpus"]).records[:3])[["id", "title", "date", "tokens", "chunk_text"]])

### 🟦 Baseline: embeddings and semantic search

Dense retrieval embeds every chunk once (cached in memory) and ranks chunks by cosine
similarity to the embedded question. The baseline model is `all-MiniLM-L6-v2`.

✏️ **Stage 2 link:** plug in the embedding model you trained or fine-tuned in Stage 2 with
`configure_embedder(...)` — see A3.2.

In [ ]:
from anatoolbox.corpus import configure_embedder

configure_embedder(None)  # None = the baseline model, all-MiniLM-L6-v2

SAMPLE_QUESTIONS = {  # ← a few questions of your own, for looking at results by eye
    "Agentic Web": ["What is the Model Context Protocol, and who supports it?", "What security risks do AI browser agents create?"],
    "Hardware & Infrastructure": ["How are export controls affecting Nvidia's sales in China?", "Which companies are building their own AI chips?"],
    "Foundation Models": ["Which open-weight models did DeepSeek release?", "How do reasoning models differ from earlier LLMs?"],
}[CONFIG["track"]]

started = time.perf_counter()
dense = retrieve.run({"query": SAMPLE_QUESTIONS[0], "input": chunks, "strategy": "dense", "size": 5}, context=ctx)
print(f"embedded {chunks['chunks']:,} chunks and searched in {time.perf_counter() - started:.0f} s")
display(pd.DataFrame(dense["passages"])[["rank", "score", "date", "title", "snippet"]])

## A2 · Evaluate search (7.2.3)

### Qualitative evaluation

Before measuring anything, look. For each sample question, do the top results answer it? Where
keyword search (sparse) and semantic search (dense) disagree, which one is right, and why?

In [ ]:
for question in SAMPLE_QUESTIONS:
    side_by_side = {}
    for strategy in ("sparse", "dense"):
        result = retrieve.run({"query": question, "input": chunks, "strategy": strategy, "size": 5}, context=ctx)
        side_by_side[strategy] = [f"{p['date']} · {str(p['title'])[:70]}" for p in result["passages"]]
    display(Markdown(f"**{question}**"))
    display(pd.DataFrame(side_by_side))

### A test set: draft, review, reuse

Quantitative evaluation needs questions whose answers — and source articles — are known.
Following the brief, `extract_test_questions` samples passages from your track and has the
**evaluation model** draft Q&A pairs of mixed types (factual, temporal, relational, analytical,
comparative). Each pair keeps the article it came from (`source_ids`), the passage
(`passage_id`) and a reference answer.

**The draft is not your test set yet.** The brief requires manual review:

1. Open `qa_draft.json` in the results folder.
2. Fix unclear questions and wrong answers; delete unanswerable, trivial or duplicate pairs;
   add questions of your own (with the `source_ids` of the articles that answer them).
3. Save the result as `qa_reviewed.json` next to it.

From then on this cell loads your reviewed file instead of drafting a new one.

**Keep in mind:** generated questions tend to reuse the passage's words, which favours keyword
search, and other articles may answer a question too — so a retrieved article that is not the
source is not necessarily wrong.

In [ ]:
DRAFT_FILE, REVIEWED_FILE = RESULTS_DIR / "qa_draft.json", RESULTS_DIR / "qa_reviewed.json"

if REVIEWED_FILE.exists():
    EVAL_SET = json.loads(REVIEWED_FILE.read_text())
    print(f"loaded {len(EVAL_SET)} reviewed questions from {REVIEWED_FILE.name}")
elif EVAL_READY:
    started = time.perf_counter()
    test_set = draft_questions.run(
        {"input": chunks, "sample_size": CONFIG["qa_passages"], "questions_per_record": CONFIG["questions_per_passage"], "min_chars": 400, "seed": 42},
        context=ctx,
    )
    EVAL_SET = test_set["questions"]
    DRAFT_FILE.write_text(json.dumps(EVAL_SET, indent=2, ensure_ascii=False))
    print(f"drafted {len(EVAL_SET)} questions from {test_set['records_sampled']} passages in {time.perf_counter() - started:.0f} s "
          f"with {test_set['model']} → {DRAFT_FILE.name}")
    print("⚠ Not reviewed yet — review it and save qa_reviewed.json before you report results.")
else:
    EVAL_SET = []
    print("No evaluation model configured and no reviewed test set — the evaluation cells are skipped.")

if EVAL_SET:
    print("question types:", pd.Series([q.get("type") for q in EVAL_SET]).value_counts(dropna=False).to_dict())
    display(pd.DataFrame(EVAL_SET)[["id", "type", "question", "reference_answer", "date"]].head(8))

### 📏 Quantitative evaluation

For each question, the source article should be retrieved. `calculate_retrieval_metrics` reports:

- **precision@k** — share of the top k results that are relevant,
- **recall@k** — share of the relevant articles found in the top k,
- **hit@k** — whether any relevant article is in the top k,
- **MRR** — mean of 1 / rank of the first relevant article.

Chunks count as their article, once. `evaluate_retrieval` runs one retrieval configuration over
the whole test set and adds a row to the comparison table — every later optimization uses it.

In [ ]:
RETRIEVAL_RESULTS = []   # one row per configuration → the comparison table in part C
RETRIEVAL_METRICS = {}   # configuration → full metrics, including per-question results
K = [1, 3, 5, 10]


def evaluate_retrieval(label, search):
    # `search(question) -> result` of retrieve_passages or rerank_passages, run for every test question.
    # Adds a row to RETRIEVAL_RESULTS and the full metrics to RETRIEVAL_METRICS[label].
    if not EVAL_SET:
        print(f"skipped {label!r}: no test set")
        return
    started = time.perf_counter()
    runs = [{"id": q["id"], "question": q["question"], "relevant": q["source_ids"], "retrieved": search(q["question"])} for q in EVAL_SET]
    metrics = retrieval_metrics.run({"results": runs, "k": K}, context=ctx)
    last = runs[-1]["retrieved"]["provenance"]
    RETRIEVAL_RESULTS.append({"configuration": label, "tool": last["tool"], **metrics["summary"],
                              "seconds": round(time.perf_counter() - started, 1), "run": last["run_id"]})
    RETRIEVAL_METRICS[label] = metrics


def by_question_type(metrics, columns=("hit@1", "hit@5", "recall@10", "reciprocal_rank")):
    types = {q["id"]: q.get("type") for q in EVAL_SET}
    per_question = pd.DataFrame(metrics["per_question"]).assign(type=lambda d: d["id"].map(types))
    return per_question.groupby("type")[list(columns)].mean().round(3).assign(questions=per_question.groupby("type").size())


evaluate_retrieval(
    "🟦 dense (baseline)",
    lambda q: retrieve.run({"query": q, "input": chunks, "strategy": "dense", "size": 30}, context=ctx),
)
if RETRIEVAL_RESULTS:
    display(pd.DataFrame(RETRIEVAL_RESULTS))
    display(by_question_type(RETRIEVAL_METRICS["🟦 dense (baseline)"]))

### Real-world monitoring

A test set is a snapshot. Once people use a system, their questions drift away from any test set.
Log real queries, the results clicked or cited, and questions that got no good answer, and turn
the failures into new test questions. For the project, it is enough to describe how you would
monitor your system.

## A3 · Optimize your search system (7.2.4)

One block per optimization in the chapter. Each ends with an `evaluate_retrieval(...)` call, so
its numbers join the comparison table. You don't have to do all of them; pick the ones your
evaluation points to.

### ✏️ A3.1 · Advanced chunking

Fixed windows cut sentences and mix topics. Structural chunking (sentences, paragraphs),
semantic chunking (split where the topic shifts) and contextual chunking (add what a chunk
lost — see `contextualize=` and `register_contextualizer`) are yours to build. Override `split`:

In [ ]:
from anatoolbox.preprocess.chunk.chunk_by_size import ChunkBySizeTool

show_hooks(ChunkBySizeTool)

In [ ]:
class MyChunker(ChunkBySizeTool):
    tool_name = "chunk_by_my_method"   # ← name your method (keep the "chunk_" prefix)
    description = "…"                  # ← one sentence on what it does

    def split(self, text, settings):
        # ✏️ YOUR TURN: return the chunks of one article, in order, as [{"text": ...}, ...].
        # settings holds the arguments (size, overlap, …); add your own via settings().
        return super().split(text, settings)   # ← the baseline, until you replace it


my_chunks = MyChunker().run({"input": articles, "size": 200, "overlap": 40}, context=ctx)
print(f"{my_chunks['chunks']:,} chunks · words per chunk: {my_chunks['tokens']}")
evaluate_retrieval(
    "✏️ dense on MyChunker",
    lambda q: retrieve.run({"query": q, "input": my_chunks, "strategy": "dense", "size": 30}, context=ctx),
)

### ✏️ A3.2 · Fine-tuning the embedding model

A general-purpose embedding model does not know that "A2A" and "Agent2Agent" are the same
thing, or which differences matter in your track. Plug in the model you fine-tuned in Stage 2 —
any function from a list of texts to vectors works. Changing the embedder re-embeds the corpus.

In [ ]:
# from sentence_transformers import SentenceTransformer
# stage2_model = SentenceTransformer("path/to/your/stage2/model")
# configure_embedder(lambda texts: stage2_model.encode(texts, normalize_embeddings=True))
# evaluate_retrieval("✏️ dense with the Stage 2 model",
#                    lambda q: retrieve.run({"query": q, "input": chunks, "strategy": "dense", "size": 30}, context=ctx))
# configure_embedder(None)   # back to the baseline model for the rest of the notebook

### 🟩 A3.3 · Integrating lexical search for precision — worked example

Embeddings capture meaning but blur exact names, version numbers and acronyms — exactly what
news questions contain. Keyword search (BM25) matches them literally. Three ways to combine the two:

1. **sparse** — BM25 alone, shipped;
2. **hybrid** — reciprocal rank fusion of the BM25 and dense *rankings*, shipped;
3. **weighted hybrid** — a weighted sum of the normalized *scores*, `α · dense + (1 − α) · BM25`.
   Not shipped — built below as a subclass, to show the full pattern: a new name, a new argument,
   one overridden hook, and the inherited filters.

In [ ]:
for strategy in ("sparse", "hybrid"):
    evaluate_retrieval(
        f"🟦 {strategy}",
        lambda q, strategy=strategy: retrieve.run({"query": q, "input": chunks, "strategy": strategy, "size": 30}, context=ctx),
    )

In [ ]:
from anatoolbox.corpus import ensure_bm25, ensure_embeddings, get_embedder
from anatoolbox.gather.retrieve.retrieve_passages import RetrievePassagesTool
from anatoolbox.retrieval import Hit, dense_rank
from anatoolbox.tool import with_properties


class RetrieveWeightedHybrid(RetrievePassagesTool):
    '''Hybrid search as a weighted sum of min-max normalized BM25 and embedding scores.'''

    # 1 · A new name (same prefix) and a description.
    tool_name = "retrieve_passages_weighted_hybrid"
    description = "Rank passages by alpha * dense score + (1 - alpha) * BM25 score, both normalized to 0-1."
    strategies = ("weighted_hybrid",)

    # 2 · A new argument: declared in the input schema, read and checked in settings().
    input_schema = with_properties(RetrievePassagesTool.input_schema, {
        "alpha": {"type": "number", "minimum": 0, "maximum": 1, "description": "Weight of the dense score."},
    })

    def settings(self, args):
        alpha = float(args.get("alpha", 0.5))
        if not 0 <= alpha <= 1:
            raise self.input_error(f"alpha must be between 0 and 1, got {alpha}.", argument="alpha")
        return {**super().settings({**args, "strategy": "weighted_hybrid"}), "alpha": alpha}

    # 3 · The one step that changes: how a query ranks the corpus.
    def rank(self, query, corpus, settings, limit):
        sparse = {hit.index: hit.score for hit in ensure_bm25(corpus).rank(query)}
        dense = {hit.index: hit.score for hit in dense_rank(get_embedder()([query]), ensure_embeddings(corpus))}
        sparse, dense = normalize(sparse), normalize(dense)
        alpha = settings["alpha"]
        combined = {i: alpha * dense.get(i, 0.0) + (1 - alpha) * sparse.get(i, 0.0) for i in sparse.keys() | dense.keys()}

        hits = []
        for index, score in sorted(combined.items(), key=lambda item: (-item[1], item[0])):
            if self.keep(corpus.records[index], settings):   # inherited: date range and metadata filters
                hits.append(Hit(index=index, score=score, strategy="weighted_hybrid"))
                if len(hits) == limit:
                    break
        return hits


def normalize(scores):
    if not scores:
        return {}
    low, high = min(scores.values()), max(scores.values())
    return {i: (s - low) / (high - low) if high > low else 1.0 for i, s in scores.items()}


weighted = RetrieveWeightedHybrid()
for alpha in (0.25, 0.5, 0.75):
    evaluate_retrieval(
        f"🟩 weighted hybrid α={alpha}",
        lambda q, alpha=alpha: weighted.run({"query": q, "input": chunks, "alpha": alpha, "size": 30}, context=ctx),
    )
if RETRIEVAL_RESULTS:
    display(pd.DataFrame(RETRIEVAL_RESULTS).drop(columns=["run"]))

**Reading the worked example.** Every row names the tool that produced it, and each run's
provenance records `alpha` — so the table cannot mix up configurations. Two cautions for your own
comparisons:

- **Choosing α on the test set and reporting the score on the same set overstates it.** Pick α on
  one part of your questions and report on the rest.
- **With a few dozen questions, small differences are noise.** Look at per-question results
  (`RETRIEVAL_METRICS[label]["per_question"]`) before calling a winner.

### ✏️ A3.4 · Leveraging metadata

Every article has a date and a publishing domain. Trend questions care about *when*; some
sources are more reliable than others. Shipped: `date_from` / `date_to`, `filters` and
`recency_half_life_days`. Custom rules go into `keep` (which records may be returned) and `boost`
(a score multiplier).

In [ ]:
show_hooks(RetrievePassagesTool)

evaluate_retrieval(
    "🟦 dense + recency (half-life 180 days)",
    lambda q: retrieve.run({"query": q, "input": chunks, "strategy": "dense", "size": 30, "recency_half_life_days": 180}, context=ctx),
)


class MyMetadataRetriever(RetrievePassagesTool):
    tool_name = "retrieve_passages_with_my_metadata_rules"   # ← name your method

    def keep(self, record, settings):
        # ✏️ YOUR TURN — e.g. exclude domains you don't trust, or restrict to a period the question names.
        return super().keep(record, settings)

    def boost(self, record, settings):
        # ✏️ YOUR TURN — e.g. favour recent articles for "latest"/"new" questions. 1.0 = unchanged.
        return super().boost(record, settings)


evaluate_retrieval(
    "✏️ dense + MyMetadataRetriever",
    lambda q: MyMetadataRetriever().run({"query": q, "input": chunks, "strategy": "dense", "size": 30}, context=ctx),
)

### A3.5 · Using reranking to address information loss

An embedding compresses a whole chunk into one vector, and some meaning is lost on the way.
A reranker reads the question and each candidate's full text together and re-scores it — too
slow for the whole corpus, fine for the top 30. Shipped: a cross-encoder. ✏️ To try another
scorer (a different cross-encoder, or a language model judging relevance), override `score`.

In [ ]:
from anatoolbox.gather.rerank.rerank_passages import RerankPassagesTool

evaluate_retrieval(
    "🟦 dense → rerank (cross-encoder)",
    lambda q: rerank.run({"input": retrieve.run({"query": q, "input": chunks, "strategy": "dense", "size": 30}, context=ctx), "keep": 10}, context=ctx),
)


class MyReranker(RerankPassagesTool):
    tool_name = "rerank_passages_with_my_scorer"   # ← name your method

    def score(self, query, texts, settings):
        # ✏️ YOUR TURN: one relevance score per text, higher = more relevant.
        return super().score(query, texts, settings)


# evaluate_retrieval("✏️ dense → MyReranker",
#                    lambda q: MyReranker().run({"input": retrieve.run({"query": q, "input": chunks, "strategy": "dense", "size": 30}, context=ctx), "keep": 10}, context=ctx))

### 📏 Search: all configurations

Pick the retrieval configuration you carry into part B. Write down why.

In [ ]:
if RETRIEVAL_RESULTS:
    table = pd.DataFrame(RETRIEVAL_RESULTS).set_index("configuration")
    display(table[["tool", "hit@1", "hit@5", "recall@10", "precision@5", "mrr", "seconds"]].round(3))

---
# Part B · End-to-end RAG (Chapter 7.3)

## B1 · A basic RAG setup (7.3.1)

### Selecting a language model

Your RAG model is set in `CONFIG["rag_model"]`: any OpenAI-compatible endpoint works, including
a compact open model you serve yourself. The model used is recorded with every answer. Answer
quality, speed and cost differ widely between models — model choice is a design decision to
justify, not a default.

### 🟦 Baseline: a basic RAG prompt

`synthesize_answer` numbers the retrieved passages `[S1]…[Sn]` with their titles and dates and
asks for an answer grounded in them, citing labels. Its citation checks do not trust the model:
**unknown citations** (labels never given), **uncited sources**, and **citation coverage** (the
share of sentences with a citation).

The baseline system: dense retrieval of 30 chunks → at most 6 passages, 2 per article → answer.

In [ ]:
from anatoolbox.enrich.synthesize.synthesize_answer import SynthesizeAnswerTool

print(synthesize.system_prompt(synthesize.settings({"question": "…", "model": "any"})))

ANSWER_ARGS = {"max_passages": 6, "max_per_source": 2, "max_chars_per_passage": 1200, "max_tokens": 400}
if LLM_READY:
    started = time.perf_counter()
    example = synthesize.run(
        {"question": SAMPLE_QUESTIONS[0], "input": retrieve.run({"query": SAMPLE_QUESTIONS[0], "input": chunks, "strategy": "dense", "size": 30}, context=ctx), **ANSWER_ARGS},
        context=ctx,
    )
    display(Markdown(f"**{SAMPLE_QUESTIONS[0]}** · {example['model']} · {time.perf_counter() - started:.0f} s\n\n{example['answer_markdown']}"))
    print("cited:", example["cited"], "| unknown:", example["unknown_citations"], "| coverage:", example["citation_coverage"])

## B2 · Evaluating your RAG system (7.3.2)

### Component-level evaluation

Part A evaluated retrieval alone. To evaluate **generation alone**, hold retrieval fixed: the
passages for each test question are retrieved once, below, and every generation variant in B3
answers from exactly the same passages. A difference in scores is then due to generation.

### 📏 End-to-end evaluation

The **evaluation model** judges each answer against its sources and the reference answer, on the
criteria the brief names — **faithfulness**, **relevance**, **context grounding** (passages and
graph evidence), **temporal grounding** (null when the question is not about time) — plus
**correctness** against the reference answer. Scores are 1–5 with a rationale.

A judge is a model too. `review_flags` marks judgments that contradict themselves; read those,
and a sample of the others, before trusting averages.

In [ ]:
RAG_RESULTS = []   # one row per (configuration, question) → the comparison tables in part C
RAG_SET = EVAL_SET[: CONFIG["rag_eval_questions"]]
CRITERIA = ["faithfulness", "relevance", "context_grounding", "temporal_grounding", "correctness"]

# Retrieval held fixed for generation variants: the baseline retrieval, once per question.
RETRIEVED = {q["id"]: retrieve.run({"query": q["question"], "input": chunks, "strategy": "dense", "size": 30}, context=ctx) for q in RAG_SET}


def evaluate_rag(label, answer_fn):
    # `answer_fn(question_dict) -> result of synthesize_answer`, judged for every question in RAG_SET.
    if not (RAG_SET and LLM_READY and EVAL_READY):
        print(f"skipped {label!r}: needs a test set, a RAG model and an evaluation model")
        return None
    started, rows = time.perf_counter(), []
    for q in RAG_SET:
        answer = answer_fn(q)
        judged = judge.run({"answer": answer, "reference_answer": q["reference_answer"]}, context=ctx)
        rows.append({
            "configuration": label, "id": q["id"], "type": q.get("type"), "question": q["question"],
            **judged["scores"], "citation_coverage": answer["citation_coverage"],
            "unknown_citations": len(answer["unknown_citations"]), "graph_facts": len(answer.get("facts") or []),
            "review_flags": "; ".join(judged["review_flags"]), "answer": answer["answer"],
            "tool": answer["provenance"]["tool"], "run": answer["provenance"]["run_id"],
        })
    RAG_RESULTS.extend(rows)
    frame = pd.DataFrame(rows)
    print(f"{label}: {len(rows)} answers judged in {time.perf_counter() - started:.0f} s")
    return frame


def show_rag(frame, columns=("faithfulness", "relevance", "context_grounding", "temporal_grounding", "correctness", "citation_coverage")):
    # Mean scores of one evaluate_rag(...) run, if it ran.
    if frame is not None:
        display(frame[list(columns)].mean().round(2).to_frame("mean").T)


def baseline_rag(q):
    return synthesize.run({"question": q["question"], "input": RETRIEVED[q["id"]], **ANSWER_ARGS}, context=ctx)


baseline_rag_results = evaluate_rag("🟦 text-only RAG (baseline)", baseline_rag)
if baseline_rag_results is not None:
    display(baseline_rag_results[CRITERIA + ["citation_coverage"]].mean().round(2).to_frame("mean").T)
    flagged = baseline_rag_results[baseline_rag_results["review_flags"] != ""]
    print(f"{len(flagged)} of {len(baseline_rag_results)} judgments flagged for review")
    display(baseline_rag_results[["id", "type", "question", "answer", "review_flags"]].head(5))

## B3 · Optimizing your RAG system (7.3.3)

### ✏️ B3.1 · Analyzing and enhancing the user query

Users ask compound, vague or broad questions. Shipped: `rewrite_query_for_retrieval` —
`decompose` into parts, `expand` into phrasings, `clarify` — with the rewrites fused into one
retrieval. Other ideas: classify the question type and route it (temporal questions → date
filters or recency), or write a hypothetical answer and embed that instead of the question.

A rewrite changes retrieval, so it is measured twice: with the retrieval metrics, and end to end.

In [ ]:
from anatoolbox.gather.rewrite.rewrite_query_for_retrieval import RewriteQueryForRetrievalTool

show_hooks(RewriteQueryForRetrievalTool)


class MyQueryRewriter(RewriteQueryForRetrievalTool):
    tool_name = "rewrite_query_with_my_method"   # ← name your method

    def rewrite(self, settings, context):
        # ✏️ YOUR TURN: return {"queries": [...], "exact_terms": [...], "time_range": {...}}.
        return super().rewrite(settings, context)


query_rewriter = MyQueryRewriter()


def rewritten_retrieval(question):
    rewrites = query_rewriter.run({"question": question, "strategy": "decompose"}, context=ctx)
    return retrieve.run({"query": question, "queries_input": rewrites, "input": chunks, "strategy": "dense", "size": 30}, context=ctx)


if LLM_READY:
    evaluate_retrieval("✏️ dense + query rewriting", rewritten_retrieval)
    show_rag(evaluate_rag("✏️ query rewriting", lambda q: synthesize.run({"question": q["question"], "input": rewritten_retrieval(q["question"]), **ANSWER_ARGS}, context=ctx)))

### ✏️ B3.2 · Optimizing the prompt

The prompt is where you set constraints (answer only from the sources, say when they are not
enough, use publication dates), the format, and how the model should reason — for example step by
step, or with a draft it then checks against the sources. Retrieval stays fixed here, so the
comparison isolates the prompt.

In [ ]:
show_hooks(SynthesizeAnswerTool)


class MyPromptedAnswer(SynthesizeAnswerTool):
    tool_name = "synthesize_answer_with_my_prompt"   # ← name your method

    def system_prompt(self, settings):
        # ✏️ YOUR TURN — replace or extend the instructions.
        return super().system_prompt(settings)

    def generate(self, system, user, settings, context):
        # ✏️ Optional — e.g. draft an answer, then ask the model to check it against the sources.
        return super().generate(system, user, settings, context)


my_prompted = MyPromptedAnswer()
show_rag(evaluate_rag("✏️ my prompt", lambda q: my_prompted.run({"question": q["question"], "input": RETRIEVED[q["id"]], **ANSWER_ARGS}, context=ctx)))

### ✏️ B3.3 · Efficient augmentation and context curation — including your knowledge graph

What reaches the prompt matters as much as what is retrieved. Shipped curation: duplicates are
dropped, the most relevant passages go to the start and end of the prompt, and `max_per_source`
limits passages per article. Your options: compress passages to their relevant sentences, order
them by date for trend questions, or fuse evidence from several retrievals.

**Graph-enhanced RAG (required by the brief).** Your Stage 1 graph adds structured evidence:
`synthesize_answer(facts=[...])` shows graph facts as `[G1]…[Gn]`, cited and checked like
passages, and the judge sees them too. Which facts to add, and whether the graph should also
steer retrieval, is the method you design and compare with the text-only baseline.

`graph_evidence` below is only a **starting point**: graph entities named in the question,
matched literally, and the facts one hop around them. Things to improve:

- **entity linking** — aliases, abbreviations, fuzzy matching (`graph.find_entities` matches exact names only);
- **relation selection** — use your schema to pick the relations that fit the question type;
- **graph-aware retrieval** — boost passages from the articles behind the facts
  (`fact_source_ids`), e.g. in a `RetrievePassagesTool` subclass whose `boost` uses them;
- **time** — prefer facts dated near the period the question asks about.

Report how many questions got any graph evidence at all: with no facts, the "graph-enhanced"
answer is just the text-only answer.

In [ ]:
from anatoolbox.graph import describe_fact, fact_source_ids


class MyCuratedAnswer(SynthesizeAnswerTool):
    tool_name = "synthesize_answer_with_my_curation"   # ← name your method

    def curate(self, passages, settings):
        # ✏️ YOUR TURN — choose, shorten and order passages. Keep labels S1…Sn (super() assigns them).
        return super().curate(passages, settings)


def graph_evidence(question, max_facts=15):
    # ✏️ YOUR TURN — the graph facts to answer `question` with, most relevant first.
    entities = GRAPH.find_entities(question)
    return GRAPH.subgraph(entities, hops=1, max_facts=max_facts)


if RAG_SET:
    coverage = [len(graph_evidence(q["question"])) for q in RAG_SET]
    print(f"graph evidence for {sum(n > 0 for n in coverage)} of {len(coverage)} questions ({'placeholder graph' if GRAPH_IS_PLACEHOLDER else 'your Stage 1 graph'})")
    example_facts = graph_evidence(RAG_SET[0]["question"])
    print("example:", RAG_SET[0]["question"], "→", len(example_facts), "facts")
    for fact in example_facts[:5]:
        print("  ", describe_fact(fact), "· articles:", fact.get("source_ids", [])[:3])

show_rag(evaluate_rag("✏️ my context curation", lambda q: MyCuratedAnswer().run({"question": q["question"], "input": RETRIEVED[q["id"]], **ANSWER_ARGS}, context=ctx)))
show_rag(evaluate_rag(
    "✏️ graph-enhanced RAG" + (" (placeholder graph)" if GRAPH_IS_PLACEHOLDER else ""),
    lambda q: synthesize.run({"question": q["question"], "input": RETRIEVED[q["id"]], "facts": graph_evidence(q["question"]), **ANSWER_ARGS}, context=ctx),
))

### B3.4 · Fine-tuning the LLM for domain-specific knowledge

Fine-tuning teaches a model vocabulary, style and task format — not reliably new facts, which is
what retrieval is for. If you fine-tuned a compact model in Stage 2 (e.g. with LoRA), serve it
through an OpenAI-compatible server such as vLLM or Ollama, point `CONFIG["base_url"]` and
`CONFIG["rag_model"]` at it, and rerun part B: the comparison tables will show whether it helps
here. Otherwise, discuss when it would be worth the cost.

---
# Part C · Results

### 📏 Retrieval: every configuration

In [ ]:
if RETRIEVAL_RESULTS:
    retrieval_table = pd.DataFrame(RETRIEVAL_RESULTS).set_index("configuration")
    display(retrieval_table[["tool", "hit@1", "hit@5", "recall@10", "precision@5", "mrr"]].round(3))
    retrieval_table.to_csv(RESULTS_DIR / "retrieval_results.csv")

### 📏 Answers: every configuration, by criterion and by question type

The brief asks for results **across question categories**, and for **text-only vs graph-enhanced
RAG** side by side.

In [ ]:
if RAG_RESULTS:
    rag = pd.DataFrame(RAG_RESULTS)
    rag.to_csv(RESULTS_DIR / "rag_results.csv", index=False)
    display(rag.groupby("configuration", sort=False)[CRITERIA + ["citation_coverage", "graph_facts"]].mean().round(2))
    display(rag.pivot_table(index="type", columns="configuration", values="faithfulness", aggfunc="mean", sort=False).round(2))
    print(f"{(rag['review_flags'] != '').sum()} of {len(rag)} judgments carry review flags — read them before reporting.")

### One limitation → one enhancement

The brief asks you to identify one limitation from your evaluation and implement one enhancement.

- **Limitation observed:** *which configuration, which question types, which criterion — with numbers from the tables above.*
- **Likely cause:** *what in the pipeline produces it — check the provenance of a few failing answers.*
- **Enhancement:** *what you changed (which block, which tool subclass) and why it should help.*
- **Result:** *before/after, on the same questions and the same judge model.*

In [ ]:
# Provenance of the inputs, and the run id of every evaluated configuration: enough to trace
# any number in the tables above back to the tool and settings that produced it.
(RESULTS_DIR / "provenance.json").write_text(json.dumps(
    {"articles": articles["provenance"], "chunks": chunks["provenance"],
     "retrieval_runs": RETRIEVAL_RESULTS, "rag_runs": [{k: r[k] for k in ("configuration", "id", "tool", "run")} for r in RAG_RESULTS]},
    indent=2, default=str,
))
print("saved in the results folder:", sorted(p.name for p in RESULTS_DIR.iterdir()))